# Adaptive Voter Model Analysis

This notebook analyzes the Adaptive Voter Model with homophily-driven rewiring, tracking:
- Opinion dynamics and convergence
- Network clustering and homophily evolution
- Component formation (both total and opinion-homogeneous)
- Polarization metrics (traditional and new index)

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from avm_two_state_v2 import (
    simulate_jit, 
    calculate_individual_homophily,
    calculate_global_clustering_coefficient
)
import pickle
import os
from joblib import Parallel, delayed

# Set random seed for reproducibility
np.random.seed(42)

## Plot Settings

In [ ]:
"""
Configure matplotlib styling for publication-quality figures.

Color palette uses colorblind-friendly colors based on:
- Wong, B. (2011). Color blindness. Nature Methods 8, 441.

Three cycler types:
1. line_cycler: Lines with markers for data series
2. marker_cycler: Markers only for scatter plots  
3. line_only_cycler: Lines without markers for smooth curves
"""
from cycler import cycler

# Define color and line style cyclers using colorblind-friendly palette
line_cycler = (cycler(color=["#E69F00", "#56B4E9", "#009E73", "#0072B2", "#D55E00", "#CC79A7", "#F0E442", "#5d4000"]) +
               cycler(linestyle=["-", "--", "-.", ":", "-", "--", "-.", "-"]))

marker_cycler = (cycler(color=["#E69F00", "#56B4E9", "#009E73", "#0072B2", "#D55E00", "#CC79A7", "#F0E442", "#5d4000"]) +
                 cycler(linestyle=["none"] * 8) +
                 cycler(marker=["o", "s", "v", "^", "<", ">", "x", "D"]))

line_only_cycler = (cycler(color=["#E69F00", "#56B4E9", "#009E73", "#0072B2", "#D55E00", "#CC79A7", "#F0E442", "#5d4000"]) +
                 cycler(linestyle=["-", "--", "-.", ":", (0, (3, 1, 1, 1)), (0, (5, 1)), (0, (3, 5, 1, 5)), (0, (1, 1))]) +
                 cycler(marker=["none"] * 8))

# Matplotlib's standard cycler for comparison
standard_cycler = cycler("color", ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", 
                                   "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"])

# Apply publication-ready matplotlib settings
plt.rc("font", family="serif", size=18)              # Serif fonts for academic style
plt.rc("savefig", dpi=200)                           # High resolution for publications
plt.rc("legend", loc="best", fontsize="medium", 
       fancybox=True, framealpha=0.5)                # Transparent legend styling
plt.rc("lines", linewidth=2.8, markersize=7, 
       markeredgewidth=2.4)                          # Bold lines for clarity
plt.rc("axes", prop_cycle=line_only_cycler)          # Default to line-only style

## Single Simulation Run

Run a single simulation to examine the time evolution of various metrics.

In [ ]:
# Create initial network
N = 100
k = 4
G = nx.erdos_renyi_graph(N, k/(N-1))
adj_matrix = nx.to_numpy_array(G)

# Initialize opinions (50/50 split)
opinions_init = np.random.choice([0, 1], size=N)

# Run simulation with tracking enabled
steps = 100000
phi = 0.5
search = 'global'

results = simulate_jit(
    adj_matrix=adj_matrix.copy(),
    opinions_init=opinions_init,
    steps=steps,
    search=search,
    phi=phi,
    directed=False,
    force_replacement=True,
    track=True,
    ignore=True,
    n_steps=1000,
    homo_prob=1.0
)

# Unpack results
(m_diff, final_adj_matrix, final_opinions, convergence_time,
 clustering_history, individual_homophily_history, group_homophily_history,
 total_homophily_history, component_history, scc_history, average_opinion_history,
 homogeneous_component_history, polarization_history) = results

print(f"Convergence time: {convergence_time:.2f}")
print(f"Final magnetization: {m_diff:.4f}")
print(f"Final homogeneous components: {homogeneous_component_history[-1]:.0f}")
print(f"Final polarization index: {polarization_history[-1]:.4f}")

## Visualization of Single Run

Plot the time evolution of key metrics from the single simulation.

In [ ]:
# Create time axis
time_steps = np.arange(len(clustering_history)) * 1000 / N

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Clustering coefficient
axes[0, 0].plot(time_steps, clustering_history, 'b-', linewidth=2)
axes[0, 0].set_xlabel('Time (updates per node)')
axes[0, 0].set_ylabel('Clustering Coefficient')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Homophily
axes[0, 1].plot(time_steps, individual_homophily_history, 'g-', linewidth=2, label='Individual')
axes[0, 1].plot(time_steps, total_homophily_history, 'r--', linewidth=2, label='Total')
axes[0, 1].set_xlabel('Time (updates per node)')
axes[0, 1].set_ylabel('Homophily')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Components
axes[1, 0].plot(time_steps, component_history, 'purple', linewidth=2, label='Total Components')
axes[1, 0].plot(time_steps, homogeneous_component_history, 'orange', linewidth=2, label='Homogeneous Components')
axes[1, 0].set_xlabel('Time (updates per node)')
axes[1, 0].set_ylabel('Number of Components')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Polarization
axes[1, 1].plot(time_steps, polarization_history, 'm-', linewidth=2)
axes[1, 1].set_xlabel('Time (updates per node)')
axes[1, 1].set_ylabel('Polarization Index')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0, 1])

plt.tight_layout()
plt.savefig('single_run_evolution.png', dpi=300, bbox_inches='tight')
plt.show()

## Comprehensive Parameter Sweep Setup

Setup for extensive parameter sweeps including homophily probability variations.

In [ ]:
# Create storage directory
store_dir = "Sweep_HK_Undirected_Cluster_Test"
os.makedirs(store_dir, exist_ok=True)

# Sweep parameters
num_run = 100
graph_type = "HK"
search_val_list = ['global', 'local']
draw_val = False
force_val = True
directed_val = True
ignore = True
phi_list = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
homo_list = [1.0]
num_agents = 100
k_val_list = [2, 3, 4, 5, 10, 20]
steps_val = 1000000
n_steps = 100

print("Parameter sweep configuration:")
print(f"  Runs per configuration: {num_run}")
print(f"  Search strategies: {search_val_list}")
print(f"  Phi values: {phi_list}")
print(f"  Homophily probabilities: {homo_list}")
print(f"  Network sizes (k): {k_val_list}")
print(f"  Maximum steps: {steps_val}")
print(f"  Measurement interval: {n_steps}")

In [ ]:
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt

# Parameters
N = 100  # Number of nodes
k = 4    # Each node connected to k nearest neighbors
p_values = np.logspace(-3, 0, 20)  # Rewiring probabilities from 0.001 to 1
n_run = 100  # Number of runs per p value

# Calculate clustering coefficients
clustering_coeffs = []
clustering_stds = []
clustering_coeffs2 = []
clustering_stds2 = []

for p in p_values:
    C_runs = []
    C_runs2 = []
    for _ in range(n_run):
        G = nx.powerlaw_cluster_graph(N, m=k // 2, p=p)
        G2 = nx.watts_strogatz_graph(N, k, p)
        C = nx.average_clustering(G)
        C2 = nx.average_clustering(G2)
        C_runs.append(C)
        C_runs2.append(C2)
    clustering_coeffs.append(np.mean(C_runs))
    clustering_stds.append(np.std(C_runs))
    clustering_coeffs2.append(np.mean(C_runs2))
    clustering_stds2.append(np.std(C_runs2))

# Plot
plt.figure(figsize=(8, 5))
plt.errorbar(p_values, clustering_coeffs, yerr=clustering_stds, fmt='o-', 
             linewidth=2, markersize=6, capsize=4, alpha=0.8, label='Powerlaw Cluster')
plt.errorbar(p_values, clustering_coeffs2, yerr=clustering_stds2, fmt='s-', 
             linewidth=2, markersize=6, capsize=4, alpha=0.8, label='Watts-Strogatz')
plt.xscale('log')
plt.xlabel('Rewiring Probability (p)', fontsize=16)
plt.ylabel('Clustering Coefficient (C)', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=14)
plt.tight_layout()
plt.show()

print(f"C at p=0 (ring lattice): {clustering_coeffs[0]:.4f} ± {clustering_stds[0]:.4f}")
print(f"C at p=1 (random graph): {clustering_coeffs[-1]:.4f} ± {clustering_stds[-1]:.4f}")

## Single Run Processing Function

Define function to process individual simulation runs with comprehensive metric tracking.

In [ ]:
def process_single_run_enhanced(run_idx, phi_i, search_val, k_val, homo_val, graph_type="WS"):
    """
    Process a single simulation run with comprehensive metric tracking.
    
    Tracks all metrics including:
    - Convergence properties
    - Degree distributions (initial and final)
    - Clustering coefficients
    - Homophily measures (individual and total)
    - Component structures (total and homogeneous)
    - Polarization index
    - Complete time evolution histories
    """
    # Initialize network
    if graph_type == "WS":
        graph_initial = nx.watts_strogatz_graph(num_agents, k=k_val, p=0.05)
        graph_adj_matrix = nx.to_numpy_array(graph_initial)
    elif graph_type == "ER":
        p_val = k_val / (num_agents - 1)
        graph_initial = nx.erdos_renyi_graph(num_agents, p=p_val)
        graph_adj_matrix = nx.to_numpy_array(graph_initial)
    elif graph_type == "BA":
        graph_initial = nx.barabasi_albert_graph(num_agents, m=k_val // 2)
        graph_adj_matrix = nx.to_numpy_array(graph_initial)
    elif graph_type == "HK":
        graph_initial = nx.powerlaw_cluster_graph(num_agents, m=k_val // 2, p=1)
        graph_adj_matrix = nx.to_numpy_array(graph_initial)
    else:
        raise ValueError("Unsupported graph type.")
    
    # Initialize opinions (50/50 split)
    opinions_init = np.array([0] * (num_agents // 2) + [1] * (num_agents // 2))
    np.random.shuffle(opinions_init)

    # Calculate initial metrics
    initial_degrees = np.array([d for n, d in graph_initial.degree()])
    initial_individual_homophily = calculate_individual_homophily(graph_adj_matrix, opinions_init)
    initial_individual_homophily = np.ma.masked_invalid(initial_individual_homophily)
    initial_degree_dist = np.bincount(initial_degrees, minlength=num_agents)
    initial_clustering = calculate_global_clustering_coefficient(graph_adj_matrix)
    
    # Run simulation
    result = simulate_jit(
        graph_adj_matrix, opinions_init,
        steps_val, search_val, phi_i, directed_val, force_val,
        track=True, ignore=ignore, n_steps=n_steps, homo_prob=homo_val
    )
    
    # Unpack all results including new metrics
    (magnetization_diff, adj_matrix_final, opinions_final, convergence_time,
     clustering_history, individual_homophily_history, group_homophily_history,
     total_homophily_history, component_history, scc_history, average_opinion_history,
     homogeneous_component_history, polarization_history) = result

    # Analyze final network state
    graph_final = nx.from_numpy_array(adj_matrix_final)
    
    # Calculate final metrics
    final_degrees = np.array([d for n, d in graph_final.degree()])
    final_degree_dist = np.bincount(final_degrees, minlength=num_agents)
    final_individual_homophily = calculate_individual_homophily(adj_matrix_final, opinions_final)
    final_individual_homophily = np.ma.masked_invalid(final_individual_homophily)
    final_clustering = calculate_global_clustering_coefficient(adj_matrix_final)
    
    # Component analysis
    components = list(nx.connected_components(graph_final))
    wcc_sizes = [len(c) for c in components]
    wcc_dist = np.bincount(wcc_sizes, minlength=num_agents+1)[1:]
    
    # For undirected graphs, SCC = WCC
    scc_sizes = wcc_sizes
    scc_dist = wcc_dist
    
    # Compile comprehensive results
    run_result = {
        # Basic convergence metrics
        'convergence_time': convergence_time,
        'converged': convergence_time != -1,
        'magnetization': abs(magnetization_diff),
        'homo_val': homo_val,
        
        # Initial state metrics
        'initial_degree_dist': initial_degree_dist,
        'initial_clustering': initial_clustering,
        'initial_homophily_dist': initial_individual_homophily,
        
        # Final state metrics
        'final_degree_dist': final_degree_dist,
        'final_clustering': final_clustering,
        'final_homophily_dist': final_individual_homophily,
        'final_polarization': polarization_history[-1] if polarization_history is not None else 0,
        'final_homogeneous_components': homogeneous_component_history[-1] if homogeneous_component_history is not None else 0,
        
        # Component distributions
        'wcc_sizes': wcc_sizes,
        'wcc_dist': wcc_dist,
        'scc_sizes': scc_sizes,
        'scc_dist': scc_dist,
        
        # Complete time evolution histories
        'clustering_history': clustering_history,
        'individual_homophily_history': individual_homophily_history,
        'total_homophily_history': total_homophily_history,
        'component_history': component_history,
        'homogeneous_component_history': homogeneous_component_history,
        'polarization_history': polarization_history,
        'average_opinion_history': average_opinion_history
    }
    
    return run_result

## Aggregate Statistics Function

Define function to aggregate statistics across multiple runs.

In [ ]:
def process_sweep_aggregation(phi_i, homo_i, search_val, num_run, k_val, graph_type="WS"):
    """
    Process all runs for a single phi and homo_prob combination and aggregate statistics.
    """
    print(f"Running simulation for phi={phi_i}, homo_prop={homo_i}, k={k_val}")
    
    # Run all simulations for this parameter combination
    run_results = [
        process_single_run_enhanced(run_idx, phi_i, search_val, k_val, homo_i, graph_type)
        for run_idx in range(num_run)
    ]
    
    # Calculate basic convergence statistics
    n_converged = sum(r['converged'] for r in run_results)
    converged_times = [r['convergence_time'] for r in run_results if r['converged']]
    
    # Initialize aggregated statistics dictionary
    stats = {
        'phi_i': phi_i,
        'homo_i': homo_i,
        'k_val': k_val,
        
        # Convergence statistics
        'convergence_probability': n_converged / num_run,
        'avg_convergence_time': np.mean(converged_times) if converged_times else -1,
        'std_convergence_time': np.std(converged_times) if converged_times else 0,
        'avg_convergence_time_all': np.mean([r['convergence_time'] for r in run_results if r['convergence_time'] != -1]),
        
        # Magnetization statistics
        'avg_magnetization': np.mean([r['magnetization'] for r in run_results]),
        'std_magnetization': np.std([r['magnetization'] for r in run_results]),
        
        # Initial clustering
        'avg_initial_clustering': np.mean([r['initial_clustering'] for r in run_results]),
        'std_initial_clustering': np.std([r['initial_clustering'] for r in run_results]),
        
        # Final clustering
        'avg_final_clustering': np.mean([r['final_clustering'] for r in run_results]),
        'std_final_clustering': np.std([r['final_clustering'] for r in run_results]),
        
        # NEW: Polarization statistics
        'avg_final_polarization': np.mean([r['final_polarization'] for r in run_results]),
        'std_final_polarization': np.std([r['final_polarization'] for r in run_results]),
        
        # NEW: Homogeneous component statistics
        'avg_final_homogeneous_components': np.mean([r['final_homogeneous_components'] for r in run_results]),
        'std_final_homogeneous_components': np.std([r['final_homogeneous_components'] for r in run_results]),
    }

    # Aggregate degree distributions
    initial_degree_dists = np.array([r['initial_degree_dist'] for r in run_results])
    stats['avg_initial_degree_dist'] = np.mean(initial_degree_dists, axis=0)
    stats['std_initial_degree_dist'] = np.std(initial_degree_dists, axis=0)

    final_degree_dists = np.array([r['final_degree_dist'] for r in run_results])
    stats['avg_final_degree_dist'] = np.mean(final_degree_dists, axis=0)
    stats['std_final_degree_dist'] = np.std(final_degree_dists, axis=0)

    # Aggregate homophily distributions
    initial_homophily_dists = np.array([r['initial_homophily_dist'] for r in run_results])
    stats['avg_initial_homophily_dist'] = np.mean(initial_homophily_dists, axis=0)
    stats['std_initial_homophily_dist'] = np.std(initial_homophily_dists, axis=0)

    final_homophily_dists = np.array([r['final_homophily_dist'] for r in run_results])
    stats['avg_final_homophily_dist'] = np.mean(final_homophily_dists, axis=0)
    stats['std_final_homophily_dist'] = np.std(final_homophily_dists, axis=0)

    # Aggregate component distributions
    wcc_dists = np.array([r['wcc_dist'] for r in run_results])
    stats['avg_wcc_dist'] = np.mean(wcc_dists, axis=0)
    stats['std_wcc_dist'] = np.std(wcc_dists, axis=0)
    
    scc_dists = np.array([r['scc_dist'] for r in run_results])
    stats['avg_scc_dist'] = np.mean(scc_dists, axis=0)
    stats['std_scc_dist'] = np.std(scc_dists, axis=0)
    
    # Aggregate time evolution histories
    if run_results[0]['clustering_history'] is not None:
        min_length = min(len(r['clustering_history']) for r in run_results)
        stats['avg_clustering_history'] = np.mean([r['clustering_history'][:min_length] for r in run_results], axis=0)
        stats['std_clustering_history'] = np.std([r['clustering_history'][:min_length] for r in run_results], axis=0)
        
    if run_results[0]['individual_homophily_history'] is not None:
        min_length = min(len(r['individual_homophily_history']) for r in run_results)
        stats['avg_ind_homophily_history'] = np.mean([r['individual_homophily_history'][:min_length] for r in run_results], axis=0)
        stats['std_ind_homophily_history'] = np.std([r['individual_homophily_history'][:min_length] for r in run_results], axis=0)
        
    if run_results[0]['total_homophily_history'] is not None:
        min_length = min(len(r['total_homophily_history']) for r in run_results)
        stats['avg_total_homophily_history'] = np.mean([r['total_homophily_history'][:min_length] for r in run_results], axis=0)
        stats['std_total_homophily_history'] = np.std([r['total_homophily_history'][:min_length] for r in run_results], axis=0)
        
    if run_results[0]['component_history'] is not None:
        min_length = min(len(r['component_history']) for r in run_results)
        stats['avg_component_history'] = np.mean([r['component_history'][:min_length] for r in run_results], axis=0)
        stats['std_component_history'] = np.std([r['component_history'][:min_length] for r in run_results], axis=0)
    
    # NEW: Aggregate homogeneous component history
    if run_results[0]['homogeneous_component_history'] is not None:
        min_length = min(len(r['homogeneous_component_history']) for r in run_results)
        stats['avg_homogeneous_component_history'] = np.mean([r['homogeneous_component_history'][:min_length] for r in run_results], axis=0)
        stats['std_homogeneous_component_history'] = np.std([r['homogeneous_component_history'][:min_length] for r in run_results], axis=0)
    
    # NEW: Aggregate polarization history
    if run_results[0]['polarization_history'] is not None:
        min_length = min(len(r['polarization_history']) for r in run_results)
        stats['avg_polarization_history'] = np.mean([r['polarization_history'][:min_length] for r in run_results], axis=0)
        stats['std_polarization_history'] = np.std([r['polarization_history'][:min_length] for r in run_results], axis=0)
    
    # Aggregate average opinion history
    if run_results[0]['average_opinion_history'] is not None:
        min_length = min(len(r['average_opinion_history']) for r in run_results)
        stats['avg_opinion_history'] = np.mean([r['average_opinion_history'][:min_length] for r in run_results], axis=0)
        stats['std_opinion_history'] = np.std([r['average_opinion_history'][:min_length] for r in run_results], axis=0)

    return stats

## Execute Comprehensive Parameter Sweep

Run the full parameter sweep with parallel processing.

In [ ]:
# Main execution for comprehensive sweep
results_enhanced = {}

for search_val in search_val_list:
    results_enhanced[search_val] = {}
    
    for k_val in k_val_list:
        print(f"\n{'='*60}")
        print(f"Running comprehensive sweep for search strategy: {search_val}, k={k_val}")
        print(f"{'='*60}")
        
        # Create parameter combinations for phi and homo_prob
        param_combinations = [(phi_i, homo_i) for phi_i in phi_list for homo_i in homo_list]
        
        # Run parallel sweep
        sweep_results = Parallel(n_jobs=5)(
            delayed(process_sweep_aggregation)(phi_i, homo_i, search_val, num_run, k_val, graph_type)
            for phi_i, homo_i in param_combinations
        )
        
        # Organize results by homo_prob values
        results_enhanced[search_val][k_val] = {}
        
        for homo_i in homo_list:
            # Filter results for this homo_prob value
            homo_results = [r for r in sweep_results if r['homo_i'] == homo_i]
            
            results_enhanced[search_val][k_val][homo_i] = {
                "phi_list": phi_list,
                "homo_val": homo_i,
                
                # Convergence metrics
                "convergence_probability": np.array([r['convergence_probability'] for r in homo_results]),
                "avg_convergence_times": np.array([r['avg_convergence_time'] for r in homo_results]),
                "std_convergence_times": np.array([r['std_convergence_time'] for r in homo_results]),
                "avg_convergence_times_all": np.array([r['avg_convergence_time_all'] for r in homo_results]),
                
                # Magnetization
                "avg_magnetization": np.array([r['avg_magnetization'] for r in homo_results]),
                "std_magnetization": np.array([r['std_magnetization'] for r in homo_results]),
                
                # Clustering metrics
                "avg_initial_clustering": np.array([r['avg_initial_clustering'] for r in homo_results]),
                "std_initial_clustering": np.array([r['std_initial_clustering'] for r in homo_results]),
                "avg_final_clustering": np.array([r['avg_final_clustering'] for r in homo_results]),
                "std_final_clustering": np.array([r['std_final_clustering'] for r in homo_results]),
                
                # NEW: Polarization metrics
                "avg_final_polarization": np.array([r['avg_final_polarization'] for r in homo_results]),
                "std_final_polarization": np.array([r['std_final_polarization'] for r in homo_results]),
                
                # NEW: Homogeneous component metrics
                "avg_final_homogeneous_components": np.array([r['avg_final_homogeneous_components'] for r in homo_results]),
                "std_final_homogeneous_components": np.array([r['std_final_homogeneous_components'] for r in homo_results]),
                
                # Component distributions
                "avg_wcc_dist": {r['phi_i']: r['avg_wcc_dist'] for r in homo_results},
                "std_wcc_dist": {r['phi_i']: r['std_wcc_dist'] for r in homo_results},
                "avg_scc_dist": {r['phi_i']: r['avg_scc_dist'] for r in homo_results},
                "std_scc_dist": {r['phi_i']: r['std_scc_dist'] for r in homo_results},
                
                # Time evolution data
                "avg_clustering_history": {r['phi_i']: r['avg_clustering_history'] for r in homo_results},
                "std_clustering_history": {r['phi_i']: r['std_clustering_history'] for r in homo_results},
                "avg_ind_homophily_history": {r['phi_i']: r['avg_ind_homophily_history'] for r in homo_results},
                "std_ind_homophily_history": {r['phi_i']: r['std_ind_homophily_history'] for r in homo_results},
                "avg_total_homophily_history": {r['phi_i']: r['avg_total_homophily_history'] for r in homo_results},
                "std_total_homophily_history": {r['phi_i']: r['std_total_homophily_history'] for r in homo_results},
                "avg_component_history": {r['phi_i']: r['avg_component_history'] for r in homo_results},
                "std_component_history": {r['phi_i']: r['std_component_history'] for r in homo_results},
                
                # NEW: Homogeneous component evolution
                "avg_homogeneous_component_history": {r['phi_i']: r['avg_homogeneous_component_history'] for r in homo_results},
                "std_homogeneous_component_history": {r['phi_i']: r['std_homogeneous_component_history'] for r in homo_results},
                
                # NEW: Polarization evolution
                "avg_polarization_history": {r['phi_i']: r['avg_polarization_history'] for r in homo_results},
                "std_polarization_history": {r['phi_i']: r['std_polarization_history'] for r in homo_results},
                
                # Opinion evolution
                "avg_opinion_history": {r['phi_i']: r['avg_opinion_history'] for r in homo_results},
                "std_opinion_history": {r['phi_i']: r['std_opinion_history'] for r in homo_results},
                
                # Degree distributions
                "avg_initial_degree_dist": {r['phi_i']: r['avg_initial_degree_dist'] for r in homo_results},
                "std_initial_degree_dist": {r['phi_i']: r['std_initial_degree_dist'] for r in homo_results},
                "avg_final_degree_dist": {r['phi_i']: r['avg_final_degree_dist'] for r in homo_results},
                "std_final_degree_dist": {r['phi_i']: r['std_final_degree_dist'] for r in homo_results},
                
                # Homophily distributions
                "avg_initial_homophily_dist": {r['phi_i']: r['avg_initial_homophily_dist'] for r in homo_results},
                "std_initial_homophily_dist": {r['phi_i']: r['std_initial_homophily_dist'] for r in homo_results},
                "avg_final_homophily_dist": {r['phi_i']: r['avg_final_homophily_dist'] for r in homo_results},
                "std_final_homophily_dist": {r['phi_i']: r['std_final_homophily_dist'] for r in homo_results},
            }

    # Save results for this search strategy
    results_filename = os.path.join(store_dir, f"results_{search_val}_enhanced_sweep_n{num_agents}.pkl")
    with open(results_filename, 'wb') as f:
        pickle.dump(results_enhanced[search_val], f)
    print(f"\nEnhanced sweep results saved to {results_filename}")

print("\n" + "="*60)
print("Comprehensive parameter sweep completed!")
print("="*60)
print(f"Results structure: results_enhanced[search_val][k_val][homo_val]")
print(f"Available homo_prop values: {homo_list}")
print(f"Available search strategies: {search_val_list}")
print(f"Available k values: {k_val_list}")
print(f"\nNew metrics added:")
print("  - avg_final_polarization / std_final_polarization")
print("  - avg_final_homogeneous_components / std_final_homogeneous_components")
print("  - avg_polarization_history / std_polarization_history")
print("  - avg_homogeneous_component_history / std_homogeneous_component_history")

# Load results

In [ ]:
# Configuration for loading results
LOAD_FROM_FILE = True  # Set to True to load from saved files instead of using current sweep
NETWORK_TYPE = "HK"  # Options: "ER", "WS", "BA", "HK"
STORAGE_DIR = "Sweep_HK_Undirected_Cluster_Test"  # Directory where results are stored

# Load results from pickle files if enabled
if LOAD_FROM_FILE:
    print(f"Loading results for {NETWORK_TYPE} network from {STORAGE_DIR}...")
    
    results_enhanced = {}
    
    for search_val in search_val_list:
        results_filename = os.path.join(STORAGE_DIR, f"results_{search_val}_enhanced_sweep_n{num_agents}.pkl")
        
        if os.path.exists(results_filename):
            with open(results_filename, 'rb') as f:
                results_enhanced[search_val] = pickle.load(f)
            print(f"  ✓ Loaded {search_val} search results from {results_filename}")
        else:
            print(f"  ✗ File not found: {results_filename}")
            print(f"    Skipping {search_val} search strategy")
    
    if results_enhanced:
        # Extract available parameters from loaded data
        sample_search = list(results_enhanced.keys())[0]
        sample_k = list(results_enhanced[sample_search].keys())[0]
        sample_homo = list(results_enhanced[sample_search][sample_k].keys())[0]
        
        k_val_list = list(results_enhanced[sample_search].keys())
        homo_list = list(results_enhanced[sample_search][sample_k].keys())
        phi_list = results_enhanced[sample_search][sample_k][sample_homo]["phi_list"]
        
        print(f"\nLoaded data summary:")
        print(f"  Network type: {NETWORK_TYPE}")
        print(f"  Search strategies: {list(results_enhanced.keys())}")
        print(f"  K values: {k_val_list}")
        print(f"  Homophily probabilities: {homo_list}")
        print(f"  Phi values: {phi_list}")
        print(f"\n✓ Results loaded successfully!")
    else:
        print("\n✗ No results loaded. Check file paths and try again.")
else:
    print("Using results from current parameter sweep (LOAD_FROM_FILE = False)")

# Degree Distribution Analysis

Visualize and analyze the degree distributions for different rewiring strategies and parameters.

In [ ]:
def plot_combined_degree_distributions(results_global, results_local, k_val, homo_val=0.9,
                                      network_type="ER", phi_values=[0, 0.5, 1.0], 
                                      label_position='top right', log_x=False, log_y=False):
    """
    Plot combined degree distributions for global and local rewiring strategies.
    
    Parameters:
    -----------
    results_global : dict
        Results dictionary for global search strategy
    results_local : dict
        Results dictionary for local search strategy
    k_val : int
        Average degree value
    homo_val : float
        Homophily probability value to analyze
    network_type : str
        Network type label (e.g., 'ER', 'WS', 'BA')
    phi_values : list
        List of phi values to plot
    label_position : str
        Position for network type label
    log_x : bool
        If True, use log scale for x-axis
    log_y : bool
        If True, use log scale for y-axis
    """
    
    plt.rc('font', family='serif')
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    def get_color_for_phi(phi_val, rewiring_type):
        """Get color based on phi value and rewiring type."""
        if phi_val == 0.0:
            return 'black'
        elif rewiring_type == 'global':
            green_colors = {
                0.1: '#90EE90', 0.2: '#9ACD32', 0.3: '#228B22', 0.4: '#006400',
                0.5: "#00BE00", 0.6: '#9ACD32', 0.7: '#7CFC00', 0.8: '#00FF00',
                0.9: '#ADFF2F', 1.0: "#3B6B53"
            }
            return green_colors.get(phi_val, '#008000')
        else:  # local rewiring
            red_colors = {
                0.1: '#FFB6C1', 0.2: '#FF69B4', 0.3: '#FF1493', 0.4: '#DC143C',
                0.5: '#B22222', 0.6: '#FF0000', 0.7: '#8B0000', 0.8: '#CD5C5C',
                0.9: '#F08080', 1.0: '#FF4500'
            }
            return red_colors.get(phi_val, '#B22222')
    
    markers = ['s', '^', 'D', 'v', '<', 'o', 'P', '*', 'X', 'h']
    
    # Plot global rewiring final distributions
    for i, phi_i in enumerate(phi_values):
        if phi_i not in phi_list:
            continue
            
        final_degree_dist = results_global[k_val][homo_val]["avg_final_degree_dist"][phi_i]
        final_degree_dist_norm = [x / num_agents for x in final_degree_dist]
        final_degree_std = results_global[k_val][homo_val]["std_final_degree_dist"][phi_i]
        final_degree_std_norm = [x / num_agents for x in final_degree_std]
        
        degrees = np.arange(len(final_degree_dist))
        
        # Filter out zero frequency values
        mask = np.array(final_degree_dist_norm) > 0
        filtered_degrees = degrees[mask]
        filtered_dist = np.array(final_degree_dist_norm)[mask]
        filtered_std = np.array(final_degree_std_norm)[mask]
        
        if len(filtered_degrees) > 0:
            # Adjust error bars for log scale
            if log_y:
                rel_error = np.where(filtered_dist > 0, filtered_std / filtered_dist, 0)
                rel_error = np.minimum(rel_error, 0.9)
                
                y_upper = filtered_dist * (1 + rel_error)
                y_lower = filtered_dist * np.maximum(1 - rel_error, 0.01 * filtered_dist)
                
                yerr_lower = filtered_dist - y_lower
                yerr_upper = y_upper - filtered_dist
                yerr = [yerr_lower, yerr_upper]
            else:
                yerr = filtered_std
            
            color = get_color_for_phi(phi_i, 'global')
            
            axes[0].errorbar(filtered_degrees, filtered_dist, 
                           yerr=yerr, marker=markers[i % len(markers)], 
                           markersize=6, linestyle='', linewidth=2,
                           color=color, alpha=0.8, capsize=3,
                           capthick=1, elinewidth=1,
                           label=rf'$p_g = {phi_i}$')
    
    axes[0].set_xlabel(r'$k_i$', fontsize=20)
    axes[0].set_ylabel(r'$P(k_i)$', fontsize=20)
    axes[0].grid(True, alpha=0.3)
    
    if log_x:
        axes[0].set_xscale('log')
        axes[0].set_xlim(0.8, num_agents)
    else:
        axes[0].set_xlim(-0.5, 50)
        
    if log_y:
        axes[0].set_yscale('log')
        #axes[0].set_yscale('symlog', linthresh=1e-5)
        axes[0].set_ylim(5e-6, 1.5)
    else:
        axes[0].set_ylim(-0.01, 0.55)

    axes[0].legend(loc='center right', bbox_to_anchor=(0.5, 0.42, 0.5, 0.5))

    # Plot local rewiring final distributions
    for i, phi_i in enumerate(phi_values):
        if phi_i not in phi_list:
            continue
            
        final_degree_dist = results_local[k_val][homo_val]["avg_final_degree_dist"][phi_i]
        final_degree_dist_norm = [x / num_agents for x in final_degree_dist]
        final_degree_std = results_local[k_val][homo_val]["std_final_degree_dist"][phi_i]
        final_degree_std_norm = [x / num_agents for x in final_degree_std]
        
        degrees = np.arange(len(final_degree_dist))
        
        mask = np.array(final_degree_dist_norm) > 0
        filtered_degrees = degrees[mask]
        filtered_dist = np.array(final_degree_dist_norm)[mask]
        filtered_std = np.array(final_degree_std_norm)[mask]
        
        if len(filtered_degrees) > 0:
            if log_y:
                rel_error = np.where(filtered_dist > 0, filtered_std / filtered_dist, 0)
                rel_error = np.minimum(rel_error, 0.9)
                
                y_upper = filtered_dist * (1 + rel_error)
                y_lower = filtered_dist * np.maximum(1 - rel_error, 0.01 * filtered_dist)
                
                yerr_lower = filtered_dist - y_lower
                yerr_upper = y_upper - filtered_dist
                yerr = [yerr_lower, yerr_upper]
            else:
                yerr = filtered_std
            
            color = get_color_for_phi(phi_i, 'local')
            
            axes[1].errorbar(filtered_degrees, filtered_dist, 
                           yerr=yerr, marker=markers[i % len(markers)], 
                           markersize=6, linestyle='', linewidth=2,
                           color=color, alpha=0.8, capsize=3,
                           capthick=1, elinewidth=1,
                           label=rf'$p_l = {phi_i}$')
    
    axes[1].set_xlabel(r'$k_i$', fontsize=20)
    axes[1].grid(True, alpha=0.3)
    
    if log_x:
        axes[1].set_xscale('log')
        axes[1].set_xlim(0.8, num_agents)
    else:
        axes[1].set_xlim(-0.5, 50)
        
    if log_y:
        axes[1].set_yscale('log')
        axes[1].set_ylim(5e-6, 1.5)
        #axes[1].set_ylim(0, 1.5)
    else:
        axes[1].set_ylim(-0.01, 0.55)
    
    axes[1].legend(loc='center right', bbox_to_anchor=(0.5, 0.42, 0.5, 0.5))

    # Add network type label
    network_labels = {'ER': 'ER', 'WS': 'WS', 'BA': 'BA', 'HK': 'HK'}
    label_text = network_labels.get(network_type, network_type)
    
    label_positions = {
        'top left': (0.05, 0.95), 'top right': (0.95, 0.95),
        'bottom left': (0.05, 0.05), 'bottom right': (0.95, 0.05),
        'center left': (0.05, 0.5), 'center right': (0.95, 0.5)
    }
    
    x_pos, y_pos = label_positions.get(label_position, (0.95, 0.95))
    
    axes[0].text(x_pos, y_pos, label_text, transform=axes[0].transAxes,
                fontsize=20, fontweight='bold', 
                ha='right' if 'right' in label_position else 'left',
                va='top' if 'top' in label_position else 'bottom',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    
    plt.tight_layout()
    
    # Save with informative filename
    scale_suffix = ""
    if log_x and log_y:
        scale_suffix = "loglog"
    elif log_y:
        scale_suffix = "semilog"
    elif log_x:
        scale_suffix = "logx"
    
    filename = f'degree_distributions_{network_type}_n{num_agents}_k{k_val}_homo{homo_val}_{scale_suffix}.pdf'
    plt.savefig(os.path.join(store_dir, filename), dpi=400, bbox_inches='tight')
    plt.show()
    
    print(f"Degree distribution plot saved: {filename}")

# Generate degree distribution plots
print("\n" + "="*60)
print("DEGREE DISTRIBUTION ANALYSIS")
print("="*60)

# Select homophily value to analyze
homo_val = 1.0

# Semi-log scale plot (log y-axis)
print("\nGenerating semi-log scale degree distribution plot...")
plot_combined_degree_distributions(
    results_enhanced['global'], 
    results_enhanced['local'], 
    k_val=2,
    homo_val=homo_val,
    network_type="HK", 
    phi_values=[0.0, 0.3, 0.5, 1.0],
    label_position='top right',
    log_x=False, 
    log_y=True
)

print("\nDegree distribution analysis complete!")

## Time Evolution: Homophily, Homophily and Echo Chambers

Comprehensive visualization of homophily, polarization and echo chamber dynamics over time for both global and local rewiring strategies.

In [ ]:
# Configuration
plt.rc("axes", prop_cycle=line_only_cycler)
k_val = 4
add_size = 7
homo_val = 1.0

# Set font sizes
TICK_SIZE = 15 + add_size
LABEL_SIZE = 18 + add_size
LEGEND_SIZE = 10 + add_size
LINE_WIDTH = 3

# Set up the figure with 3 rows (homophily, polarization, echo chambers)
plt.style.use('seaborn-v0_8-whitegrid')
fig, axs = plt.subplots(3, 2, figsize=(20, 12), sharex=True)

# Network type configuration
network_type = "HK"
network_labels = {'ER': 'ER', 'WS': 'WS', 'BA': 'BA', 'HK': 'HK'}
label_text = network_labels.get(network_type, network_type)

# Color configuration
cmap = plt.cm.summer
cmap2 = plt.cm.autumn
phi_samples = [0, 0.3, 0.5, 0.7, 1]
colors = [cmap(i/len(phi_samples)) for i in range(len(phi_samples))]
colors2 = [cmap2(i/len(phi_samples)) for i in range(len(phi_samples))]

# ====================
# HOMOPHILY PLOTS (Top Row)
# ====================

# Global Homophily - Top Left
for phi_i, color in zip(phi_samples, colors):
    data = results_enhanced['global'][k_val][homo_val]
    avg_homophily = data["avg_ind_homophily_history"][phi_i]
    std_homophily = data["std_ind_homophily_history"][phi_i]
    
    # Clip avg ± std to [0, 1]
    upper = np.clip(avg_homophily + std_homophily, 0, 1)
    lower = np.clip(avg_homophily - std_homophily, 0, 1)
    
    time_steps = np.arange(0, len(avg_homophily)*n_steps, n_steps)
    
    # Determine color and label
    plot_color = 'black' if phi_i == 0 else color
    
    # Plot mean line and shaded region
    axs[0,0].plot(
        time_steps, avg_homophily, 
        color=plot_color, linewidth=LINE_WIDTH, label=rf'$p_g = {phi_i:.1f}$'
    )
    axs[0,0].fill_between(
        time_steps, lower, upper,
        color=plot_color, alpha=0.2, linewidth=0
    )

# Configure global homophily subplot
axs[0,0].set_ylabel(rf'$H$', fontsize=LABEL_SIZE)
axs[0,0].set_ylim(-0.05, 1.05)
axs[0,0].set_xscale('log')
axs[0,0].tick_params(axis='both', which='major', labelsize=TICK_SIZE)
axs[0,0].legend(frameon=True, framealpha=1, fontsize=LEGEND_SIZE, loc='lower right')

# Local Homophily - Top Right
for phi_i, color in zip(phi_samples, colors2):
    data = results_enhanced['local'][k_val][homo_val]
    avg_homophily = data["avg_ind_homophily_history"][phi_i]
    std_homophily = data["std_ind_homophily_history"][phi_i]
    
    # Clip avg ± std to [0, 1]
    upper = np.clip(avg_homophily + std_homophily, 0, 1)
    lower = np.clip(avg_homophily - std_homophily, 0, 1)
    
    time_steps = np.arange(0, len(avg_homophily)*n_steps, n_steps)
    
    # Determine color and label
    plot_color = 'black' if phi_i == 0 else color
    
    # Plot mean line and shaded region
    axs[0,1].plot(
        time_steps, avg_homophily, 
        color=plot_color, linewidth=LINE_WIDTH, label=rf'$p_l = {phi_i:.1f}$'
    )
    axs[0,1].fill_between(
        time_steps, lower, upper,
        color=plot_color, alpha=0.2, linewidth=0
    )

# Configure local homophily subplot
axs[0,1].set_ylim(-0.05, 1.05)
axs[0,1].set_xscale('log')
axs[0,1].tick_params(axis='both', which='major', labelsize=TICK_SIZE)
axs[0,1].legend(frameon=True, framealpha=1, fontsize=LEGEND_SIZE, loc='lower right')

# ====================
# POLARIZATION PLOTS (Middle Row)
# ====================

# Global Polarization - Middle Left
for phi_i, color in zip(phi_samples, colors):
    data = results_enhanced['global'][k_val][homo_val]
    avg_polarization = data["avg_polarization_history"][phi_i]
    std_polarization = data["std_polarization_history"][phi_i]
    
    # Clip avg ± std to [0, 1]
    upper = np.clip(avg_polarization + std_polarization, 0, 1)
    lower = np.clip(avg_polarization - std_polarization, 0, 1)
    
    time_steps = np.arange(0, len(avg_polarization)*n_steps, n_steps)
    
    # Determine color
    plot_color = 'black' if phi_i == 0 else color
    
    # Plot mean line and shaded region
    axs[1,0].plot(
        time_steps, avg_polarization, 
        color=plot_color, linewidth=LINE_WIDTH
    )
    axs[1,0].fill_between(
        time_steps, lower, upper,
        color=plot_color, alpha=0.2, linewidth=0
    )

# Configure global polarization subplot
axs[1,0].set_ylabel(rf'$|M|$', fontsize=LABEL_SIZE)
axs[1,0].set_ylim(-0.05, 1.05)
axs[1,0].set_xscale('log')
axs[1,0].tick_params(axis='both', which='major', labelsize=TICK_SIZE)

# Local Polarization - Middle Right
for phi_i, color in zip(phi_samples, colors2):
    data = results_enhanced['local'][k_val][homo_val]
    avg_polarization = data["avg_polarization_history"][phi_i]
    std_polarization = data["std_polarization_history"][phi_i]
    
    # Clip avg ± std to [0, 1]
    upper = np.clip(avg_polarization + std_polarization, 0, 1)
    lower = np.clip(avg_polarization - std_polarization, 0, 1)
    
    time_steps = np.arange(0, len(avg_polarization)*n_steps, n_steps)
    
    # Determine color
    plot_color = 'black' if phi_i == 0 else color
    
    # Plot mean line and shaded region
    axs[1,1].plot(
        time_steps, avg_polarization, 
        color=plot_color, linewidth=LINE_WIDTH
    )
    axs[1,1].fill_between(
        time_steps, lower, upper,
        color=plot_color, alpha=0.2, linewidth=0
    )

# Configure local polarization subplot
axs[1,1].set_ylim(-0.05, 1.05)
axs[1,1].set_xscale('log')
axs[1,1].tick_params(axis='both', which='major', labelsize=TICK_SIZE)

# ====================
# ECHO CHAMBERS PLOTS (Bottom Row)
# ====================

# Global Echo Chambers - Bottom Left
for phi_i, color in zip(phi_samples, colors):
    data = results_enhanced['global'][k_val][homo_val]
    avg_homogeneous_comp = data["avg_homogeneous_component_history"][phi_i]
    std_homogeneous_comp = data["std_homogeneous_component_history"][phi_i]
    
    # Convert to arrays for processing
    avg_homogeneous_comp = np.array(avg_homogeneous_comp) + 1
    std_homogeneous_comp = np.array(std_homogeneous_comp) + 1
    
    # Clip lower bound to minimum of 1
    lower = np.clip(avg_homogeneous_comp - std_homogeneous_comp, 1, None)
    upper = avg_homogeneous_comp + std_homogeneous_comp
    
    time_steps = np.arange(0, len(avg_homogeneous_comp)*n_steps, n_steps)
    
    # Determine color
    plot_color = 'black' if phi_i == 0 else color
    
    # Plot mean line and shaded region
    axs[2,0].plot(time_steps, avg_homogeneous_comp, color=plot_color, linewidth=LINE_WIDTH)
    axs[2,0].fill_between(
        time_steps, lower, upper,
        color=plot_color, alpha=0.2, linewidth=0
    )

# Configure global echo chambers subplot
axs[2,0].set_ylabel(rf'$EC+1$', fontsize=LABEL_SIZE)
axs[2,0].set_xlabel(r'$t$', fontsize=LABEL_SIZE)
axs[2,0].set_ylim(0.5, 1.05 * num_agents)
axs[2,0].set_xscale('log')
axs[2,0].set_yscale('log')
axs[2,0].tick_params(axis='both', which='major', labelsize=TICK_SIZE)

# Local Echo Chambers - Bottom Right
for phi_i, color in zip(phi_samples, colors2):
    data = results_enhanced['local'][k_val][homo_val]
    avg_homogeneous_comp = data["avg_homogeneous_component_history"][phi_i]
    std_homogeneous_comp = data["std_homogeneous_component_history"][phi_i]
    
    # Convert to arrays for processing
    avg_homogeneous_comp = np.array(avg_homogeneous_comp) + 1
    std_homogeneous_comp = np.array(std_homogeneous_comp) + 1
    
    # Clip lower bound to minimum of 1
    lower = np.clip(avg_homogeneous_comp - std_homogeneous_comp, 1, None)
    upper = avg_homogeneous_comp + std_homogeneous_comp
    
    time_steps = np.arange(0, len(avg_homogeneous_comp)*n_steps, n_steps)
    
    # Determine color
    plot_color = 'black' if phi_i == 0 else color
    
    # Plot mean line and shaded region
    axs[2,1].plot(time_steps, avg_homogeneous_comp, color=plot_color, linewidth=LINE_WIDTH)
    axs[2,1].fill_between(
        time_steps, lower, upper+1,
        color=plot_color, alpha=0.2, linewidth=0
    )

# Configure local echo chambers subplot
axs[2,1].set_xlabel(r'$t$', fontsize=LABEL_SIZE)
axs[2,1].set_ylim(0.5, 1.05 * num_agents)
axs[2,1].set_xscale('log')
axs[2,1].set_yscale('log')
axs[2,1].tick_params(axis='both', which='major', labelsize=TICK_SIZE)

# ====================
# FINAL LAYOUT ADJUSTMENTS
# ====================

# Set consistent x-limits for all subplots
for ax in axs.flat:
    ax.set_xlim(n_steps, steps_val)

# Adjust layout and align labels
plt.tight_layout(pad=3.0)
fig.align_ylabels(axs[:,0])  # Align left column y-labels
fig.align_ylabels(axs[:,1])  # Align right column y-labels

# Save plot
filename = f"homophily_polarization_echo_chambers_{network_type}_n{num_agents}_k{k_val}_homo{homo_val}.pdf"
plt.savefig(os.path.join(store_dir, filename), bbox_inches='tight', dpi=400)
plt.show()

print(f"Time evolution plot saved: {filename}")

## Time Evolution: Polarization and Echo Chambers

Comprehensive visualization of polarization and echo chamber dynamics over time for both global and local rewiring strategies.

In [ ]:
# Configuration
plt.rc("axes", prop_cycle=line_only_cycler)
k_val = 4
add_size = 7
homo_val = 1.0

# Set font sizes
TICK_SIZE = 15 + add_size
LABEL_SIZE = 18 + add_size
LEGEND_SIZE = 8 + add_size
LINE_WIDTH = 3

# Set up the figure with 2 rows (polarization top, echo chambers bottom)
plt.style.use('seaborn-v0_8-whitegrid')
fig, axs = plt.subplots(2, 2, figsize=(20, 8), sharex=True)

# Network type configuration
network_type = "ER"
network_labels = {'ER': 'ER', 'WS': 'WS', 'BA': 'BA'}
label_text = network_labels.get(network_type, network_type)

# Color configuration
cmap = plt.cm.summer
cmap2 = plt.cm.autumn
phi_samples = [0, 0.3, 0.5, 0.7, 1]
colors = [cmap(i/len(phi_samples)) for i in range(len(phi_samples))]
colors2 = [cmap2(i/len(phi_samples)) for i in range(len(phi_samples))]

# ====================
# POLARIZATION PLOTS (Top Row)
# ====================

# Global Polarization - Top Left
for phi_i, color in zip(phi_samples, colors):
    data = results_enhanced['global'][k_val][homo_val]
    avg_polarization = data["avg_polarization_history"][phi_i]
    std_polarization = data["std_polarization_history"][phi_i]
    
    # Clip avg ± std to [0, 1]
    upper = np.clip(avg_polarization + std_polarization, 0, 1)
    lower = np.clip(avg_polarization - std_polarization, 0, 1)
    
    time_steps = np.arange(0, len(avg_polarization)*n_steps, n_steps)
    
    # Determine color and label
    plot_color = 'black' if phi_i == 0 else color
    
    # Plot mean line and shaded region
    axs[0,0].plot(
        time_steps, avg_polarization, 
        color=plot_color, linewidth=LINE_WIDTH, label=rf'$p_g = {phi_i:.1f}$'
    )
    axs[0,0].fill_between(
        time_steps, lower, upper,
        color=plot_color, alpha=0.2, linewidth=0
    )

# Configure global polarization subplot
axs[0,0].set_ylabel(rf'$P$', fontsize=LABEL_SIZE)
axs[0,0].set_ylim(-0.05, 1.05)
axs[0,0].set_xscale('log')
axs[0,0].tick_params(axis='both', which='major', labelsize=TICK_SIZE)
axs[0,0].legend(frameon=True, framealpha=1, fontsize=LEGEND_SIZE, loc='lower left')

# Local Polarization - Top Right
for phi_i, color in zip(phi_samples, colors2):
    data = results_enhanced['local'][k_val][homo_val]
    avg_polarization = data["avg_polarization_history"][phi_i]
    std_polarization = data["std_polarization_history"][phi_i]
    
    # Clip avg ± std to [0, 1]
    upper = np.clip(avg_polarization + std_polarization, 0, 1)
    lower = np.clip(avg_polarization - std_polarization, 0, 1)
    
    time_steps = np.arange(0, len(avg_polarization)*n_steps, n_steps)
    
    # Determine color and label
    plot_color = 'black' if phi_i == 0 else color
    
    # Plot mean line and shaded region
    axs[0,1].plot(
        time_steps, avg_polarization, 
        color=plot_color, linewidth=LINE_WIDTH, label=rf'$p_l = {phi_i:.1f}$'
    )
    axs[0,1].fill_between(
        time_steps, lower, upper,
        color=plot_color, alpha=0.2, linewidth=0
    )

# Configure local polarization subplot
axs[0,1].set_ylim(-0.05, 1.05)
axs[0,1].set_xscale('log')
axs[0,1].tick_params(axis='both', which='major', labelsize=TICK_SIZE)
axs[0,1].legend(frameon=True, framealpha=1, fontsize=LEGEND_SIZE, loc='lower left')

# ====================
# ECHO CHAMBERS PLOTS (Bottom Row)
# ====================

# Global Echo Chambers - Bottom Left
for phi_i, color in zip(phi_samples, colors):
    data = results_enhanced['global'][k_val][homo_val]
    avg_homogeneous_comp = data["avg_homogeneous_component_history"][phi_i]
    std_homogeneous_comp = data["std_homogeneous_component_history"][phi_i]
    
    # Convert to arrays for processing
    avg_homogeneous_comp = np.array(avg_homogeneous_comp) + 1
    std_homogeneous_comp = np.array(std_homogeneous_comp) + 1
    
    # Clip lower bound to minimum of 1
    lower = np.clip(avg_homogeneous_comp - std_homogeneous_comp, 1, None)
    upper = avg_homogeneous_comp + std_homogeneous_comp
    
    time_steps = np.arange(0, len(avg_homogeneous_comp)*n_steps, n_steps)
    
    # Determine color
    plot_color = 'black' if phi_i == 0 else color
    
    # Plot mean line and shaded region
    axs[1,0].plot(time_steps, avg_homogeneous_comp, color=plot_color, linewidth=LINE_WIDTH)
    axs[1,0].fill_between(
        time_steps, lower, upper,
        color=plot_color, alpha=0.2, linewidth=0
    )

# Configure global echo chambers subplot
axs[1,0].set_ylabel(rf'$EC+1$', fontsize=LABEL_SIZE)
axs[1,0].set_xlabel(r'$t$', fontsize=LABEL_SIZE)
axs[1,0].set_ylim(0.5, 1.05 * num_agents)
axs[1,0].set_xscale('log')
axs[1,0].set_yscale('log')
axs[1,0].tick_params(axis='both', which='major', labelsize=TICK_SIZE)

# Local Echo Chambers - Bottom Right
for phi_i, color in zip(phi_samples, colors2):
    data = results_enhanced['local'][k_val][homo_val]
    avg_homogeneous_comp = data["avg_homogeneous_component_history"][phi_i]
    std_homogeneous_comp = data["std_homogeneous_component_history"][phi_i]
    
    # Convert to arrays for processing
    avg_homogeneous_comp = np.array(avg_homogeneous_comp) + 1
    std_homogeneous_comp = np.array(std_homogeneous_comp) + 1
    
    # Clip lower bound to minimum of 1
    lower = np.clip(avg_homogeneous_comp - std_homogeneous_comp, 1, None)
    upper = avg_homogeneous_comp + std_homogeneous_comp
    
    time_steps = np.arange(0, len(avg_homogeneous_comp)*n_steps, n_steps)
    
    # Determine color
    plot_color = 'black' if phi_i == 0 else color
    
    # Plot mean line and shaded region
    axs[1,1].plot(time_steps, avg_homogeneous_comp, color=plot_color, linewidth=LINE_WIDTH)
    axs[1,1].fill_between(
        time_steps, lower, upper+1,
        color=plot_color, alpha=0.2, linewidth=0
    )

# Configure local echo chambers subplot
axs[1,1].set_xlabel(r'$t$', fontsize=LABEL_SIZE)
axs[1,1].set_ylim(0.5, 1.05 * num_agents)
axs[1,1].set_xscale('log')
axs[1,1].set_yscale('log')
axs[1,1].tick_params(axis='both', which='major', labelsize=TICK_SIZE)

# ====================
# FINAL LAYOUT ADJUSTMENTS
# ====================

# Set consistent x-limits for all subplots
for ax in axs.flat:
    ax.set_xlim(n_steps, steps_val)

# Adjust layout and align labels
plt.tight_layout(pad=3.0)
fig.align_ylabels(axs[:,0])  # Align left column y-labels
fig.align_ylabels(axs[:,1])  # Align right column y-labels

# Save plot
filename = f"polarization_echo_chambers_{network_type}_n{num_agents}_k{k_val}_homo{homo_val}.pdf"
plt.savefig(os.path.join(store_dir, filename), bbox_inches='tight', dpi=400)
plt.show()

print(f"Time evolution plot saved: {filename}")

In [ ]:
def plot_echo_chamber_distribution(results_global, results_local, k_val, homo_val, output_dir, 
                                   network_type="ER", isolated=False, label_position='top right'):
    """
    Plot echo chamber (opinion-homogeneous component) size distribution.
    
    Parameters:
    -----------
    results_global : dict
        Results for global rewiring strategy
    results_local : dict
        Results for local rewiring strategy
    k_val : int
        Average degree value
    homo_val : float
        Homophily probability
    output_dir : str
        Directory to save the plot
    network_type : str
        Network type label ('ER', 'WS', 'BA')
    isolated : bool
        If False, exclude single-node components from normalization
    label_position : str
        Position for network type label
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

    # Set font sizes
    TICK_SIZE = 24
    LABEL_SIZE = 24
    LEGEND_SIZE = 24

    # Get common phi values for both plots
    datag = results_global[k_val][homo_val]
    phi_list_full = datag["phi_list"]
    dataglen = len(phi_list_full)
    phi_list = [phi_list_full[0], phi_list_full[2], phi_list_full[int(dataglen/2)], phi_list_full[-1]]
    colors = ["#044104", "#216E39",  "#56FD8B", "#13EE13"]
    colors2 = ["#720303", "#921D1D", "#D46363",  "#FFC400"]
    
    def get_clipped_errors(freqs, errs):
        """Clip error bars to valid probability range [0, 1]."""
        freqs = np.clip(freqs, 0, 1)
        lower_err = np.minimum(errs, freqs)
        upper_err = np.minimum(errs, 1 - freqs)
        return np.vstack([lower_err, upper_err])
    
    def process_data(avg_dist, std_dist, num_agents):
        """Process and normalize distribution data, optionally excluding isolated nodes."""
        # Create mask to exclude size=1 clusters if isolated=False
        if isolated == False:
            non_isolated_mask = np.arange(len(avg_dist)) >= 1  # Skip index 0 (size 1)
        else:
            non_isolated_mask = np.arange(len(avg_dist)) >= 0  # Include all
        
        # Normalize ONLY non-isolated clusters
        total_non_isolated = np.sum(avg_dist[non_isolated_mask])
        if total_non_isolated > 0:
            avg_dist_normalized = np.zeros_like(avg_dist)
            avg_dist_normalized[non_isolated_mask] = avg_dist[non_isolated_mask] / total_non_isolated
            std_dist_normalized = std_dist / total_non_isolated
        else:
            avg_dist_normalized = np.zeros_like(avg_dist)
            std_dist_normalized = std_dist
        
        # Get indices of non-zero probabilities (excluding isolated nodes if specified)
        nonzero_indices = np.where((avg_dist_normalized > 0) & non_isolated_mask)[0]
        sizes = (nonzero_indices + 1) / num_agents  # Normalize by network size
        freqs = avg_dist_normalized[nonzero_indices]
        errs = std_dist_normalized[nonzero_indices]
        
        return sizes, freqs, errs
    
    # Plot global rewiring (left subplot)
    datag = results_global[k_val][homo_val]
    for i, phi in enumerate(phi_list):
        if phi > 0:
            sizes, freqs, errs = process_data(
                datag["avg_wcc_dist"][phi],
                datag["std_wcc_dist"][phi],
                num_agents
            )
            
            yerr = get_clipped_errors(freqs, errs)
            
            ax1.errorbar(
                sizes, freqs, 
                yerr=yerr,
                fmt='o', markersize=8, capsize=4,
                color=colors[i], alpha=0.8,
                label=rf'$p_g={np.round(phi, 1)}$'
            )
        else:
            sizes, freqs, errs = process_data(
                datag["avg_wcc_dist"][phi],
                datag["std_wcc_dist"][phi],
                num_agents
            )
            
            yerr = get_clipped_errors(freqs, errs)
            
            ax1.errorbar(
                sizes, freqs, 
                yerr=yerr,
                fmt='o', markersize=8, capsize=4,
                color='black', alpha=0.8,
                label=rf'$p_g={np.round(phi, 1)}$'
            )
    
    ax1.set_xlabel(rf'$s_i$', fontsize=LABEL_SIZE)
    ax1.set_ylabel(rf'$P(s_i)$', fontsize=LABEL_SIZE)
    ax1.set_xlim(-0.02, 1.02)
    ax1.set_ylim(-0.05, 1.05)
    ax1.grid(True, which="both", ls="--", alpha=0.3)
    ax1.legend(fontsize=LEGEND_SIZE, framealpha=0.9, loc='upper left')
    ax1.tick_params(axis='both', which='major', labelsize=TICK_SIZE)
    
    # Plot local rewiring (right subplot)
    datal = results_local[k_val][homo_val]
    for i, phi in enumerate(phi_list):
        if phi > 0:
            sizes, freqs, errs = process_data(
                datal["avg_wcc_dist"][phi],
                datal["std_wcc_dist"][phi],
                num_agents
            )
            
            yerr = get_clipped_errors(freqs, errs)
            
            ax2.errorbar(
                sizes, freqs,
                yerr=yerr,
                fmt='o', markersize=8, capsize=4,
                color=colors2[i], alpha=0.8,
                label=rf'$p_l={np.round(phi, 1)}$'
            )
        else: 
            sizes, freqs, errs = process_data(
                datal["avg_wcc_dist"][phi],
                datal["std_wcc_dist"][phi],
                num_agents
            )
            
            yerr = get_clipped_errors(freqs, errs)
            
            ax2.errorbar(
                sizes, freqs,
                yerr=yerr,
                fmt='o', markersize=8, capsize=4,
                color='black', alpha=0.8,
                label=rf'$p_l={np.round(phi, 1)}$'
            )
    
    ax2.set_xlabel(rf'$s_i$', fontsize=LABEL_SIZE)
    ax2.set_xlim(-0.02, 1.02)
    ax2.set_ylim(-0.05, 1.05)
    ax2.grid(True, which="both", ls="--", alpha=0.3)
    ax2.legend(fontsize=LEGEND_SIZE, framealpha=0.9, loc='upper left')
    ax2.tick_params(axis='both', which='major', labelsize=TICK_SIZE)
    
    # Add network type label
    network_labels = {'ER': 'ER', 'WS': 'WS', 'BA': 'BA', 'HK': 'HK'}
    label_text = network_labels.get(network_type, network_type)
    
    label_positions = {
        'top left': (0.05, 0.95),
        'top right': (0.95, 0.95),
        'bottom left': (0.05, 0.05),
        'bottom right': (0.95, 0.05),
        'center left': (0.05, 0.5),
        'center right': (0.95, 0.5)
    }
    
    x_pos, y_pos = label_positions.get(label_position, (0.95, 0.95))
    
    # Add label to left subplot
    ax1.text(0.55, y_pos, label_text, transform=ax1.transAxes,
             fontsize=34, fontweight='bold', 
             ha='right' if 'right' in label_position else 'left',
             va='top' if 'top' in label_position else 'bottom',
             bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      alpha=0.8, edgecolor='black'))

    
    plt.tight_layout()
    
    # Save with descriptive filename
    isolation_suffix = "nonisolated" if not isolated else "all"
    filename = f"echo_chamber_distribution_{isolation_suffix}_{network_type}_k{k_val}_homo{homo_val}.pdf"
    plt.savefig(os.path.join(output_dir, filename), bbox_inches='tight', dpi=400)
    plt.show()
    
    print(f"Echo chamber distribution plot saved: {filename}")

# Generate echo chamber distribution plot
print("\n" + "="*60)
print("ECHO CHAMBER DISTRIBUTION ANALYSIS")
print("="*60)

homo_val = 1.0

print("\nGenerating echo chamber distribution plot (excluding isolated nodes)...")
plot_echo_chamber_distribution(
    results_enhanced['global'], 
    results_enhanced['local'], 
    k_val=10,
    homo_val=homo_val,
    output_dir=store_dir,
    network_type="HK",
    isolated=False,
    label_position='top right'
)

print("\nEcho chamber distribution analysis complete!")

## Echo Chamber Size Distribution Histogram

Visualize the distribution of echo chamber sizes using normalized histograms with adjustable bin sizes.

In [ ]:
# Set up the figure and subplots
fig, axs = plt.subplots(2, 2, figsize=(20, 16), sharex='col', sharey='row')
plt.subplots_adjust(hspace=0.1, wspace=0.15)

# Set font sizes
size_increase = 6
TICK_SIZE = 20 + size_increase
LABEL_SIZE = 20 + size_increase
LEGEND_SIZE = 20 + size_increase
TITLE_SIZE = 22 + size_increase

# CONVERGENCE TIME COLOR SCALE OPTIONS
use_log_scale_convergence = False  # Set to False for linear scale
log_scale_min_threshold = 1e-3    # Minimum value for log scale to avoid log(0)

# Select homophily value to analyze
homo_val_heatmap = 1.0

# Add network type parameter
network_type = "HK"  # Change based on your network
network_labels = {'ER': 'ER', 'WS': 'WS', 'BA': 'BA', 'HK': 'HK'}
label_text = network_labels.get(network_type, network_type)

# Use the global homo_list for different homophily probabilities
# For heatmap, we'll use k_val and phi as axes, fixing homo_val
k_values = np.array(k_val_list)
phi_values = np.array(phi_list)

# ====================
# Top Left: Global Largest Component Size
# ====================
phi_grid, k_grid = np.meshgrid(phi_values, k_values)

largest_component_sizes = np.zeros_like(phi_grid)
for i, k_val in enumerate(k_values):
    for j, phi_val in enumerate(phi_values):
        # Get the WCC distribution for this phi value
        wcc_dist = results_enhanced['global'][k_val][homo_val_heatmap]["avg_wcc_dist"][phi_val]
        # Find the largest component size (last non-zero element)
        non_zero_indices = np.nonzero(wcc_dist)[0]
        if len(non_zero_indices) > 0:
            largest_component_size = non_zero_indices[-1] + 1  # +1 because indices start at 0
        else:
            largest_component_size = 0
        largest_component_sizes[i, j] = largest_component_size / num_agents

im1 = axs[0, 0].pcolormesh(
    phi_grid, k_grid, largest_component_sizes,
    cmap='magma',
    vmin=0, vmax=1,
    shading='auto'
)

axs[0, 0].set_xticks(phi_values)
axs[0, 0].set_yticks(k_values)
axs[0, 0].tick_params(axis='both', labelsize=TICK_SIZE)
axs[0, 0].set_ylabel(r'$k_{avg}$', fontsize=LABEL_SIZE)

# Add network type label to top left
axs[0, 0].text(0.12, 0.95, label_text, transform=axs[0, 0].transAxes,
              fontsize=28, fontweight='bold', ha='right', va='top',
              bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.9, edgecolor='black'))

# ====================
# Top Right: Local Largest Component Size
# ====================
largest_component_sizes = np.zeros_like(phi_grid)
for i, k_val in enumerate(k_values):
    for j, phi_val in enumerate(phi_values):
        # Get the WCC distribution for this phi value
        wcc_dist = results_enhanced['local'][k_val][homo_val_heatmap]["avg_wcc_dist"][phi_val]
        # Find the largest component size
        non_zero_indices = np.nonzero(wcc_dist)[0]
        if len(non_zero_indices) > 0:
            largest_component_size = non_zero_indices[-1] + 1
        else:
            largest_component_size = 0
        largest_component_sizes[i, j] = largest_component_size / num_agents

im2 = axs[0, 1].pcolormesh(
    phi_grid, k_grid, largest_component_sizes,
    cmap='magma',
    vmin=0, vmax=1,
    shading='auto'
)

axs[0, 1].set_xticks(phi_values)
axs[0, 1].set_yticks(k_values)
axs[0, 1].tick_params(axis='both', labelsize=TICK_SIZE)

# Add colorbar for largest component size plots
cbar_ax1 = fig.add_axes([0.92, 0.55, 0.02, 0.35])  # [left, bottom, width, height]
cbar1 = fig.colorbar(im1, cax=cbar_ax1)
cbar1.set_label(rf'$s_1$', fontsize=LABEL_SIZE)
cbar1.ax.tick_params(labelsize=TICK_SIZE)

# ====================
# Bottom Left: Global Convergence Time
# ====================
conv_times_global = np.zeros_like(phi_grid)
for i, k_val in enumerate(k_values):
    conv_times_global[i, :] = results_enhanced['global'][k_val][homo_val_heatmap]["avg_convergence_times"]

# ====================
# Bottom Right: Local Convergence Time  
# ====================
conv_times_local = np.zeros_like(phi_grid)
for i, k_val in enumerate(k_values):
    conv_times_local[i, :] = results_enhanced['local'][k_val][homo_val_heatmap]["avg_convergence_times"]

# ====================
# CONVERGENCE TIME COLOR SCALE SETUP
# ====================
# Collect all convergence time data
all_conv_times = np.concatenate([
    results_enhanced['global'][k][homo_val_heatmap]["avg_convergence_times"] for k in k_values
] + [
    results_enhanced['local'][k][homo_val_heatmap]["avg_convergence_times"] for k in k_values
])
# Exclude non-converged runs (-1 values)
valid_conv_times = all_conv_times[all_conv_times > 0] 

if use_log_scale_convergence:
    # Logarithmic color scale
    from matplotlib.colors import LogNorm
    
    # Set bounds for log scale
    vmin_log = max(log_scale_min_threshold, np.min(valid_conv_times))
    vmax_log = np.max(valid_conv_times)
    
    # Process convergence time data for log scale
    conv_times_global_log = np.where(conv_times_global > 0, 
                                    np.maximum(conv_times_global, log_scale_min_threshold),
                                    log_scale_min_threshold)
    conv_times_local_log = np.where(conv_times_local > 0,
                                   np.maximum(conv_times_local, log_scale_min_threshold),
                                   log_scale_min_threshold)
    
    # Create plots with logarithmic normalization
    im3 = axs[1, 0].pcolormesh(
        phi_grid, k_grid, conv_times_global_log,
        cmap='viridis',
        norm=LogNorm(vmin=vmin_log, vmax=vmax_log),
        shading='auto'
    )
    
    im4 = axs[1, 1].pcolormesh(
        phi_grid, k_grid, conv_times_local_log,
        cmap='viridis',
        norm=LogNorm(vmin=vmin_log, vmax=vmax_log),
        shading='auto'
    )
    
    colorbar_label = rf'$\tau$'
    print(f"Using logarithmic color scale: {vmin_log:.2e} to {vmax_log:.2e}")
    
else:
    # Linear color scale (original behavior)
    vmax_linear = np.max(valid_conv_times)
    
    im3 = axs[1, 0].pcolormesh(
        phi_grid, k_grid, conv_times_global,
        cmap='viridis',
        vmin=0,
        vmax=vmax_linear,
        shading='auto'
    )
    
    im4 = axs[1, 1].pcolormesh(
        phi_grid, k_grid, conv_times_local,
        cmap='viridis',
        vmin=0,
        vmax=vmax_linear,
        shading='auto'
    )
    
    colorbar_label = rf'$\tau$'
    print(f"Using linear color scale: 0 to {vmax_linear:.2f}")

# Configure subplot properties
for ax, xlabel in zip([axs[1, 0], axs[1, 1]], [r'$p_g$', rf'$p_l$']):
    ax.set_xticks(phi_values)
    ax.set_yticks(k_values)
    ax.tick_params(axis='both', labelsize=TICK_SIZE)
    ax.set_xlabel(xlabel, fontsize=LABEL_SIZE)

axs[1, 0].set_ylabel(r'$k_{avg}$', fontsize=LABEL_SIZE)

# Add colorbar for convergence time plots
cbar_ax2 = fig.add_axes([0.92, 0.1, 0.02, 0.35])  # [left, bottom, width, height]
cbar2 = fig.colorbar(im3, cax=cbar_ax2)
cbar2.set_label(colorbar_label, fontsize=LABEL_SIZE)
cbar2.ax.tick_params(labelsize=TICK_SIZE)

# Add panel labels
panel_labels = ['(a)', '(b)', '(c)', '(d)']

for i, ax in enumerate(axs.flat):
    ax.text(-0.1, 1.1, panel_labels[i], transform=ax.transAxes, 
            fontsize=TITLE_SIZE, fontweight='bold', va='top')

# Save with appropriate filename based on color scale
scale_suffix = "_logscale" if use_log_scale_convergence else "_linearscale"
filename = f"combined_heatmaps_largest_component_{network_type}_homo{homo_val_heatmap}{scale_suffix}.pdf"
plt.savefig(os.path.join(store_dir, filename), dpi=400, bbox_inches='tight')
plt.show()

print(f"\nHeatmap plot saved: {filename}")
print(f"Analyzing homophily value: {homo_val_heatmap}")

In [ ]:
# Network size sweep
N_values = [50, 100, 200, 500]
phi_fixed = 0.5
k = 4
n_realizations = 5
steps = 150000

size_results = {
    'convergence_times': np.zeros((len(N_values), n_realizations)),
    'final_homogeneous_components': np.zeros((len(N_values), n_realizations)),
    'final_polarization': np.zeros((len(N_values), n_realizations)),
    'final_homophily': np.zeros((len(N_values), n_realizations))
}

print("Starting network size sweep...")
for i, N in enumerate(N_values):
    print(f"\nN = {N}")
    for j in range(n_realizations):
        # Create network
        G = nx.erdos_renyi_graph(N, k/(N-1))
        adj_matrix = nx.to_numpy_array(G)
        opinions_init = np.random.choice([0, 1], size=N)
        
        # Run simulation
        results = simulate_jit(
            adj_matrix=adj_matrix.copy(),
            opinions_init=opinions_init,
            steps=steps,
            search='global',
            phi=phi_fixed,
            directed=False,
            force_replacement=True,
            track=True,
            ignore=True,
            n_steps=5000,
            homo_prob=1.0
        )
        
        # Unpack and store
        (m_diff, final_adj_matrix, final_opinions, convergence_time,
         clustering_history, individual_homophily_history, group_homophily_history,
         total_homophily_history, component_history, scc_history, average_opinion_history,
         homogeneous_component_history, polarization_history) = results
        
        size_results['convergence_times'][i, j] = convergence_time
        size_results['final_homogeneous_components'][i, j] = homogeneous_component_history[-1]
        size_results['final_polarization'][i, j] = polarization_history[-1]
        size_results['final_homophily'][i, j] = individual_homophily_history[-1]
        
        print(f"  Completed {j + 1}/{n_realizations} realizations")

print("\nSize sweep complete!")

## Save All Results

Save the complete analysis results for future reference.

In [ ]:
# Consolidate all results
all_results = {
    'enhanced_sweep': results_enhanced,
    'parameters': {
        'k_values': k_val_list,
        'phi_values': phi_list,
        'homo_values': homo_list,
        'n_realizations': num_run,
        'steps': steps_val,
        'n_steps': n_steps,
        'search_strategies': search_val_list
    }
}

with open(os.path.join(store_dir, 'complete_enhanced_results.pkl'), 'wb') as f:
    pickle.dump(all_results, f)

print("All enhanced results saved")
print(f"\nAnalysis Summary:")
print(f"  Phi values: {len(phi_list)} ({phi_list})")
print(f"  Homophily probabilities: {len(homo_list)} ({homo_list})")
print(f"  Search strategies: {len(search_val_list)} ({search_val_list})")
print(f"  Network sizes (k): {len(k_val_list)} ({k_val_list})")
print(f"  Runs per configuration: {num_run}")
print(f"  Total configurations: {len(phi_list) * len(homo_list) * len(search_val_list) * len(k_val_list)}")
print(f"  Total simulations: {len(phi_list) * len(homo_list) * len(search_val_list) * len(k_val_list) * num_run}")

## Load Previously Saved Results

Load pickle files from previous parameter sweeps for different network types and search strategies.